In [0]:
%run ./00_setup

In [0]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
import datetime as dt

orders_df = (
    spark.table("workspace.bronze.orders_raw")
)

In [0]:
orders_df.display()

In [0]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
import datetime as dt

context = gx.get_context()

suite_name = "orders_suite"
context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

datasource_config = {
    "name": "spark_orders_ds",
    "class_name": "Datasource",
    "execution_engine": {
        "class_name": "SparkDFExecutionEngine",
        "persist": False
    },
    "data_connectors": {
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
}

datasource = context.add_datasource(**datasource_config)

batch_request = RuntimeBatchRequest(
    datasource_name="spark_orders_ds",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="orders_asset",
    runtime_parameters={"batch_data": orders_df},
    batch_identifiers={"default_identifier_name": "orders_batch"}
)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_be_unique("order_id")
validator.expect_column_values_to_not_be_null("customer_id")

validator.expect_column_values_to_be_in_set("country", ["ES", "MX", "AR", "CO", "CL"])
validator.expect_column_values_to_be_in_set("currency", ["EUR", "MXN", "ARS", "COP", "CLP"])

validator.expect_column_values_to_be_between("quantity", min_value=1, max_value=100)
validator.expect_column_values_to_be_between("unit_price", min_value=0.01, max_value=10000)
validator.expect_column_values_to_be_between("order_total", min_value=0.01, max_value=1000000)

validator.expect_column_values_to_be_in_set("payment_method", ["CARD", "TRANSFER", "CASH", "PAYPAL"])
validator.expect_column_values_to_be_in_set("order_status", ["CREATED", "PAID", "CANCELLED", "REFUNDED"])

validator.expect_column_values_to_match_regex(
    "email",
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    mostly=0.99
)


validator.expect_column_values_to_be_between(
    "order_ts",
    min_value="2022-01-01 00:00:00",
    max_value=str(dt.datetime.utcnow())
)

validator.save_expectation_suite(discard_failed_expectations=False)
orders_result = validator.validate()

display(orders_result.to_json_dict())

save_json(
    f"{VALIDATION_PATH}/orders_validation.json",
    orders_result.to_json_dict()
)